In [21]:
import requests
import pandas as pd
import numpy as np
import os
import pprint
import re
from tqdm import tqdm
import time

In [ ]:
locgov_url_search = 'https://www.loc.gov/collections/chronicling-america/?qs=ku+klux+klan&ops=AND&searchType=advanced&dl=page&start_date=1915-01-01&end_date=1945-12-31&c=160&sp=336'

In [25]:
r = s.get(f'https://www.loc.gov/collections/chronicling-america/?qs=ku+klux+klan&ops=AND&searchType=advanced&dl=page&start_date=1915-01-01&end_date=1945-12-31&c=160&sp=1&fo=json', timeout=5)
r.raise_for_status()  # Check if the request was successful
data = r.json()

In [71]:
data['content']['results'][1]

{'access_restricted': False,
 'aka': ['http://www.loc.gov/resource/sn86074011/1921-01-28/ed-1/?sp=1'],
 'batch': ['msar_junebug_ver01'],
 'campaigns': [],
 'composite_location': ['0/united states/',
  '1/united states/mississippi/',
  '2/united states/mississippi/monroe/',
  '3/united states/mississippi/monroe/aberdeen/'],
 'contributor': ['mississippi department of archives and history'],
 'date': '1921-01-28',
 'dates': ['1921-01-28'],
 'description': ['jfllE AEECQEEIi WEEKLY V 77 TY tfYt W 7 I EVERY FRIDAY MORNING j J fll M 1 I f j f J fl f 01 Advertising rates on application OUR JH rrniTT ISCOZIPLEIE Service the Best Prices Kh Delivery Prompt Commercial Wedding scd Society Printing Execute MAIL OEDEES SOLICITE D Try Cs for Good Work and Low it VOL 42 ABERDEEN MISS JAR 28 1921 NO 27 V I Attent Ion9 Farmer WE HAVE OPENED UP AN INFORMATION BUREAU IN OUR BANK TO HELP THE FARMERS OF MONROE COUNTY LOTS OF FARMERS HAVE HAY THAT THEY WOULD LIKE to exchange for lumber if you have call on uo

In [74]:
def scrape_and_save_data(pageno):
    city = []
    state = []
    date = []
    lccn = []
    text = []
    title = []
    county = []

    try:
        r = s.get(f'https://www.loc.gov/collections/chronicling-america/?qs=ku+klux+klan&ops=AND&searchType=advanced&dl=page&start_date=1915-01-01&end_date=1945-12-31&c=160&sp={pageno}&fo=json', timeout=5)
        r.raise_for_status()  # Check if the request was successful
        data = r.json()

        for item in range(0, len(data['content']['results'])):
            city.append(data['content']['results'][item]['location_city'][0] if 'location_city' in data['content']['results'][item] else '')
            date.append(data['content']['results'][item]['date'])
            state.append(data['content']['results'][item]['location_state'][0] if 'location_state' in data['content']['results'][item] else '')
            lccn.append(data['content']['results'][item]['number_lccn'])
            text.append(data['content']['results'][item]['description'][0])
            title.append(data['content']['results'][item]['partof_title'][0])
            county.append(data['content']['results'][item]['location_county'][0] if 'location_county' in data['content']['results'][item] else '')

        df = pd.DataFrame({
            'city': city,
            'state': state,
            'date': date,
            'lCCN': lccn,
            'text': text,
            'title': title
        })

        output_dir =  '/Volumes/T7/chroniclingamerica/kkk/revival'
        os.makedirs(output_dir, exist_ok=True)
        df.to_csv(f'{output_dir}/kkk-revival-{pageno}.csv', index=False)
        time.sleep(30)
    except requests.exceptions.RequestException as e:
        # print(f"Error: {e}")
        failed_page.append(pageno)

failed_page = []
s = requests.Session()

for pageno in tqdm(range(int(np.round(53614 / 160)) + 1)): #336
# for pageno in tqdm(range(0,1)):
    no = pageno+1
    if str(no) in [i.strip('kkk-revival-').strip('.csv') for i in os.listdir('/Volumes/T7/chroniclingamerica/kkk/revival')]:
        continue
    else:
        scrape_and_save_data(no)

 68%|██████▊   | 229/336 [1:28:43<41:27, 23.25s/it]  


KeyboardInterrupt: 

In [9]:
api_query = requests.get(locgov_url_search)

In [10]:
search_result = api_query.json()

In [11]:
len(search_result)

39